# 🔧 Notebook 02 — Feature Engineering
## Predictfy × Locaweb — FIAP Challenge 2026

**Objetivo:** Transformar o dataset bruto em um dataset modelável (`incidents_features.parquet`), aplicando as decisões documentadas no [Notebook 01 — EDA](./01_eda.ipynb).

**Decisões do nb 01 aplicadas aqui:**
- Target: `KPI Violado? == SIM` (validado como consistente com Duração > OLA)
- Data leakage: remover Duração, Resolvido, Encerrado, Código de fechamento, Solução
- Nulos de Produto/Categoria: preencher com `"DESCONHECIDO"` (MNAR)
- Desbalanceamento: 248 SIM vs ~25.352 NAO — tratamento via SMOTE no nb 04
- P2 tem dados apenas a partir de 2025 no subset KPI

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

print('✅ Imports concluídos')

---
## 2. Carregar e Filtrar Subset KPI

In [ ]:
df = pd.read_excel('../data/raw/LW-DATASET.xlsx')
print(f'Dataset completo: {df.shape}')

# Filtrar apenas incidentes que entraram para KPI
kpi = df[df['Entrou para KPI?'] == 'SIM'].copy()
print(f'Subset KPI: {kpi.shape}')

# Validações
prioridades_kpi = set(kpi['Prioridade'].unique())
assert prioridades_kpi <= {'2 - Alta', '3 - Média'}, f'Prioridades inesperadas: {prioridades_kpi}'
assert 'Sem Intervenção' not in kpi['Status'].values, 'Status Sem Intervenção encontrado no KPI!'
assert 25_000 <= len(kpi) <= 26_000, f'Shape inesperado: {len(kpi)}'

print(f'✅ Validações OK')
print(f'   Prioridades: {kpi["Prioridade"].value_counts().to_dict()}')
print(f'   Status: {kpi["Status"].value_counts().to_dict()}')

---
## 3. Criação do Target

> **Data leakage (nb 01):** Duração, Resolvido, Encerrado, Código de fechamento e Solução **não podem ser features** — são informações disponíveis apenas após a resolução do incidente.

In [ ]:
# Target binário: 1 = violou OLA, 0 = não violou
kpi['target_ola'] = (kpi['KPI Violado?'] == 'SIM').astype(int)

n_pos = kpi['target_ola'].sum()
n_neg = len(kpi) - n_pos
taxa = n_pos / len(kpi) * 100

print(f'Target criado:')
print(f'  Violações (1): {n_pos}')
print(f'  Não violações (0): {n_neg}')
print(f'  Taxa de violação: {taxa:.2f}%')
print(f'  Razão: ~1:{n_neg // n_pos}')

---
## 4. Features Temporais

Extraídas da coluna `Aberto` — representam o momento em que o incidente foi aberto, informação disponível em tempo real.

In [ ]:
# Features temporais básicas
kpi['hora'] = kpi['Aberto'].dt.hour
kpi['dia_semana'] = kpi['Aberto'].dt.dayofweek  # 0=Segunda, 6=Domingo
kpi['mes'] = kpi['Aberto'].dt.month
kpi['trimestre'] = kpi['Aberto'].dt.quarter

# Features binárias de contexto temporal
kpi['is_horario_comercial'] = kpi['hora'].between(9, 17).astype(int)
kpi['is_fim_de_semana'] = (kpi['dia_semana'] >= 5).astype(int)
kpi['is_segunda_terca'] = (kpi['dia_semana'] <= 1).astype(int)

# Feature adicional: período do dia (madrugada/manhã/tarde/noite)
# Justificativa: padrões de operação variam por turno, não apenas horário comercial
def periodo_dia(h):
    if 0 <= h < 6: return 0  # madrugada
    elif 6 <= h < 12: return 1  # manhã
    elif 12 <= h < 18: return 2  # tarde
    else: return 3  # noite

kpi['periodo_dia'] = kpi['hora'].apply(periodo_dia)

# Feature adicional: dia do mês (pode captar padrões de batch/processamento mensal)
kpi['dia_mes'] = kpi['Aberto'].dt.day

# Feature: semana do ano (sazonalidade mais fina que mês)
kpi['semana_ano'] = kpi['Aberto'].dt.isocalendar().week.astype(int)

print('Features temporais criadas:')
print(f'  hora, dia_semana, mes, trimestre')
print(f'  is_horario_comercial, is_fim_de_semana, is_segunda_terca')
print(f'  periodo_dia, dia_mes, semana_ano')
print(f'  Shape atual: {kpi.shape}')

---
## 5. Features de Volume Histórico (Lags e Médias Móveis)

Lags representam o volume de incidentes em dias anteriores — informação disponível no momento da abertura do incidente. Capturam tendências de carga operacional.

In [ ]:
# Volume diário geral
kpi['data'] = kpi['Aberto'].dt.date
vol_diario = kpi.groupby('data').size().reset_index(name='vol_dia')
vol_diario['data'] = pd.to_datetime(vol_diario['data'])
vol_diario = vol_diario.sort_values('data')

# Lags e médias móveis
vol_diario['lag_1d'] = vol_diario['vol_dia'].shift(1)
vol_diario['lag_7d'] = vol_diario['vol_dia'].shift(7)
vol_diario['rolling_7d'] = vol_diario['vol_dia'].rolling(7, min_periods=1).mean().round(2)
vol_diario['rolling_30d'] = vol_diario['vol_dia'].rolling(30, min_periods=1).mean().round(2)

# Volume diário por prioridade (lags separados)
# Justificativa: um pico de P2 tem impacto diferente de um pico de P3
for prio_label, prio_name in [('2 - Alta', 'p2'), ('3 - Média', 'p3')]:
    vol_prio = kpi[kpi['Prioridade'] == prio_label].groupby('data').size().reset_index(name=f'vol_{prio_name}')
    vol_prio['data'] = pd.to_datetime(vol_prio['data'])
    vol_diario = vol_diario.merge(vol_prio, on='data', how='left')
    vol_diario[f'vol_{prio_name}'] = vol_diario[f'vol_{prio_name}'].fillna(0)
    vol_diario[f'lag_1d_{prio_name}'] = vol_diario[f'vol_{prio_name}'].shift(1)

print(f'Volume diário calculado: {vol_diario.shape}')
print(f'Período: {vol_diario["data"].min().date()} a {vol_diario["data"].max().date()}')
vol_diario.head(10)

In [ ]:
# Merge lags no dataset principal
n_antes = len(kpi)
kpi['data_dt'] = pd.to_datetime(kpi['data'])

merge_cols = ['data', 'lag_1d', 'lag_7d', 'rolling_7d', 'rolling_30d',
              'lag_1d_p2', 'lag_1d_p3']
kpi = kpi.merge(
    vol_diario[merge_cols],
    left_on='data_dt', right_on='data',
    how='left',
    suffixes=('', '_vol')
)

# Remover linhas sem lag (primeiros 7 dias)
n_sem_lag = kpi['lag_7d'].isnull().sum()
kpi = kpi.dropna(subset=['lag_1d', 'lag_7d'])
n_depois = len(kpi)

print(f'Linhas antes do merge: {n_antes}')
print(f'Linhas sem lag_7d (primeiros 7 dias): {n_sem_lag}')
print(f'Linhas após drop: {n_depois}')
print(f'Perda: {n_antes - n_depois} linhas ({(n_antes - n_depois)/n_antes*100:.2f}%)')

# Garantir que não perdemos violações demais
print(f'Violações restantes: {kpi["target_ola"].sum()}')

---
## 6. Prioridade Binária

In [ ]:
# P2 (Alta) = 1, P3 (Média) = 0
kpi['prioridade_bin'] = (kpi['Prioridade'] == '2 - Alta').astype(int)

print(f'Distribuição prioridade_bin:')
print(kpi['prioridade_bin'].value_counts().to_string())

---
## 7. Tratamento de Nulos e Encoding Categórico

**Decisão (nb 01):** Preencher nulos de Produto, Categoria e Subcategoria com `"DESCONHECIDO"` — a ausência é MNAR (não aleatória) e pode ser um sinal preditivo.

In [ ]:
# Preencher nulos com 'DESCONHECIDO'
for col in ['Produto', 'Categoria', 'Subcategoria']:
    n_null = kpi[col].isnull().sum()
    kpi[col] = kpi[col].fillna('DESCONHECIDO')
    print(f'  {col}: {n_null} nulos preenchidos ({n_null/len(kpi)*100:.1f}%)')

print()

# Label Encoding
le = LabelEncoder()
colunas_enc = ['Produto', 'Categoria', 'Subcategoria', 'Grupo designado', 'Aberto por']

for col in colunas_enc:
    kpi[f'{col}_enc'] = le.fit_transform(kpi[col].astype(str))
    n_cat = kpi[col].nunique()
    print(f'  {col}_enc: {n_cat} categorias únicas')

# Frequency Encoding alternativo para Produto e Grupo designado
# Justificativa: códigos anônimos sem ordem natural; frequência captura
# a importância relativa de cada produto/grupo no volume operacional
for col in ['Produto', 'Grupo designado']:
    freq_map = kpi[col].value_counts(normalize=True).to_dict()
    kpi[f'{col}_freq'] = kpi[col].map(freq_map).round(6)
    print(f'  {col}_freq: frequency encoding criado')

print(f'\nShape após encoding: {kpi.shape}')

---
## 8. Seleção das Features Finais

In [ ]:
FEATURES = [
    # Temporais
    'hora', 'dia_semana', 'mes', 'trimestre',
    'is_horario_comercial', 'is_fim_de_semana', 'is_segunda_terca',
    'periodo_dia', 'dia_mes', 'semana_ano',
    # Prioridade
    'prioridade_bin',
    # Lags e médias móveis
    'lag_1d', 'lag_7d', 'rolling_7d', 'rolling_30d',
    'lag_1d_p2', 'lag_1d_p3',
    # Categóricas (label encoded)
    'Produto_enc', 'Categoria_enc', 'Subcategoria_enc',
    'Grupo designado_enc', 'Aberto por_enc',
    # Categóricas (frequency encoded)
    'Produto_freq', 'Grupo designado_freq',
]

TARGET = 'target_ola'

df_model = kpi[FEATURES + [TARGET]].copy()

# Validação: zero nulos
nulos_check = df_model.isnull().sum()
nulos_check = nulos_check[nulos_check > 0]
if len(nulos_check) > 0:
    print('⚠️ Colunas com nulos:')
    print(nulos_check)
    # Preencher lags de prioridade que podem ter NaN
    for c in ['lag_1d_p2', 'lag_1d_p3']:
        df_model[c] = df_model[c].fillna(0)

assert df_model.isnull().sum().sum() == 0, 'Dataset com nulos!'

print(f'✅ Dataset final: {df_model.shape}')
print(f'   Features: {len(FEATURES)}')
print(f'   Target: {TARGET}')
print(f'   Violações (1): {df_model[TARGET].sum()}')
print(f'   Não violações (0): {(df_model[TARGET] == 0).sum()}')
print(f'   Taxa: {df_model[TARGET].mean()*100:.2f}%')

In [ ]:
# Visualização rápida das distribuições das features
fig, axes = plt.subplots(4, 6, figsize=(20, 14))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    if i < len(axes):
        axes[i].hist(df_model[col], bins=30, color='#1a73e8', edgecolor='white', alpha=0.8)
        axes[i].set_title(col, fontsize=9)
        axes[i].tick_params(labelsize=7)

for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das Features Finais', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Exportação

Salvar em Parquet (eficiente, tipado) e CSV (compatibilidade). Estes arquivos desbloqueiam os notebooks 03, 04 e 05.

In [ ]:
import pyarrow

# Exportar
df_model.to_parquet('../data/processed/incidents_features.parquet', index=False)
df_model.to_csv('../data/processed/incidents_features.csv', index=False)

# Verificação pós-export
check = pd.read_parquet('../data/processed/incidents_features.parquet')
assert check.shape == df_model.shape, f'Shape diverge: {check.shape} vs {df_model.shape}'
assert check['target_ola'].sum() == df_model['target_ola'].sum(), 'Contagem de violações diverge!'

print(f'✅ Parquet gerado: {check.shape}')
print(f'   Violações preservadas: {check["target_ola"].sum()}')
print(f'   Arquivo: ../data/processed/incidents_features.parquet')
print(f'   CSV:     ../data/processed/incidents_features.csv')

check.head()

---
## 10. Schema do Dataset Exportado

| # | Feature | Tipo | Descrição | Origem |
|---|---------|------|-----------|--------|
| 1 | `hora` | int | Hora de abertura (0–23) | `Aberto.dt.hour` |
| 2 | `dia_semana` | int | Dia da semana (0=Seg, 6=Dom) | `Aberto.dt.dayofweek` |
| 3 | `mes` | int | Mês (1–12) | `Aberto.dt.month` |
| 4 | `trimestre` | int | Trimestre (1–4) | `Aberto.dt.quarter` |
| 5 | `is_horario_comercial` | int | 1 se hora entre 9–17 | Derivada de `hora` |
| 6 | `is_fim_de_semana` | int | 1 se sábado ou domingo | Derivada de `dia_semana` |
| 7 | `is_segunda_terca` | int | 1 se segunda ou terça | Derivada de `dia_semana` |
| 8 | `periodo_dia` | int | 0=madrugada, 1=manhã, 2=tarde, 3=noite | Derivada de `hora` |
| 9 | `dia_mes` | int | Dia do mês (1–31) | `Aberto.dt.day` |
| 10 | `semana_ano` | int | Semana ISO do ano (1–53) | `Aberto.dt.isocalendar().week` |
| 11 | `prioridade_bin` | int | 1=P2 (Alta), 0=P3 (Média) | Derivada de `Prioridade` |
| 12 | `lag_1d` | float | Volume total de incidentes KPI no dia anterior | Lag sobre vol_dia |
| 13 | `lag_7d` | float | Volume total de incidentes KPI 7 dias atrás | Lag sobre vol_dia |
| 14 | `rolling_7d` | float | Média móvel 7 dias do volume diário | Rolling sobre vol_dia |
| 15 | `rolling_30d` | float | Média móvel 30 dias do volume diário | Rolling sobre vol_dia |
| 16 | `lag_1d_p2` | float | Volume de P2 no dia anterior | Lag por prioridade |
| 17 | `lag_1d_p3` | float | Volume de P3 no dia anterior | Lag por prioridade |
| 18 | `Produto_enc` | int | Label Encoding do Produto | LabelEncoder |
| 19 | `Categoria_enc` | int | Label Encoding da Categoria | LabelEncoder |
| 20 | `Subcategoria_enc` | int | Label Encoding da Subcategoria | LabelEncoder |
| 21 | `Grupo designado_enc` | int | Label Encoding do Grupo | LabelEncoder |
| 22 | `Aberto por_enc` | int | Label Encoding de Aberto por | LabelEncoder |
| 23 | `Produto_freq` | float | Frequency Encoding do Produto | value_counts(normalize) |
| 24 | `Grupo designado_freq` | float | Frequency Encoding do Grupo | value_counts(normalize) |
| 25 | **`target_ola`** | int | **TARGET** — 1=violou OLA, 0=não violou | `KPI Violado? == SIM` |

> **Total:** 24 features + 1 target = 25 colunas

---
*Dataset pronto. Próximos notebooks: 03 (Prophet), 04 (XGBoost + SMOTE), 05 (K-Means)*